# TM-RugPull dataset initial analysis

## Data collection

In [1]:
# Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull.xlsx'

data = pd.read_excel(file)

# Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

(1000, 27)


,Project Title,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Sign,first deposits,Blockchain Type,Smart Contract (online),smart Contract (offline),website,x profile,class,project starting date,project end date,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
0,HyperVerse Token (HVT),7.650000e+00,1.500000e-01,9.100000e-06,8.000000e-07,BSC,623909,25752,9.537265e-03,4.929203e-02,166600000000100,HVT,44,POSA,sourse code,CODE,https://thehyperverse.net/index.html,https://twitter.com/HyperVerse6,scam,2022-01-27,2023-07-14,148,148,29,8,74,47
1,Fintoch,1.795000e-11,1.907000e-11,1.727000e-10,1.769000e-10,BSC,"1,492,842","147,791",2.487228e+03,4.503032e+07,34403.74594,BEP-20 TOKEN*,1,POSA,sourse code,CODE,https://web.archive.org/web/20230603123631/htt...,NaN,scam,2022-07-12,2023-06-19,820,167,4530,1590,48,4
2,Flare Token,1.955000e-03,5.519000e-04,4.158000e-04,2.838000e-04,BSC,"184,694",15173,8.901714e+14,6.695544e+17,10000000000,Flare,53,POSA,sourse code,CODE,https://pipeflare.io/,https://x.com/MetaFlareToken,scam,2021-10-24,2022-11-24,421,156,6,5,1710,1150
3,Safuu Protocol,2.070000e+02,2.110000e+02,7.000000e+01,2.400000e+01,BSC,"275,530",151979,2.457331e+10,8.139816e+14,61634066.59803,SAFUU,2,POSA,sourse code,CODE,https://safuu.com/,https://x.com/safuuxofficial,scam,2022-02-03,2022-08-13,327,323,0,0,5,4
4,SCT,2.986000e-01,1.659000e-01,1.831000e-01,1.552000e-01,BSC,8445,"6,126\n",4.252172e+06,1.410127e+05,"42,896,736.739367",SCT,52,POSA,sourse code,CODE,https://supercells.jp/en/,https://x.com/scttoken,scam,2023-02-27,2024-07-09,4990000,345000,6,3,1830,594


In [2]:
# Check the balance between classes in the whole dataset
# Source: https://note.nkmk.me/en/python-pandas-value-counts/#value_counts

class_counts = data['class'].value_counts()
class_percent = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts.to_string(), "\nIn %:", class_percent.to_string())

Counts: class
scam      599
normal    401 
In %: class
scam      59.9
normal    40.1


In [3]:
# Check general info about data

print("\nDataset information:")
data.info()


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 27 columns):
 #   Column                                                 Non-Null Count  Dtype         
---  ------                                                 --------------  -----         
 0   Project Title                                          1000 non-null   object        
 1   MaxPrice (Quarter 1)                                   1000 non-null   float64       
 2   MaxPrice (Quarter 2)                                   1000 non-null   float64       
 3   MaxPrice (Quarter 3)                                   1000 non-null   float64       
 4   MaxPrice (Quarter 4)                                   1000 non-null   float64       
 5   Blockchain                                             1000 non-null   object        
 6   the number of Transactions                             1000 non-null   object        
 7   Token concentration ratio per holder            

In [4]:
# Analysis of blockchain distribution in data

blockchain_distribution = data['Blockchain'].value_counts(normalize=True) * 100
print(blockchain_distribution.to_string())

Blockchain
ETH        63.3
BSC        32.1
POLYGON     2.4
ARBI        1.2
FANTOM      0.4
CRONO       0.2
BASE        0.2
FTM         0.1
SNOW        0.1


In [5]:
# Analyse scam rate per chain

scam_rate_per_chain = (
    data.groupby('Blockchain')['class']
        .apply(lambda s: (s == 'scam').mean())
        .sort_values(ascending=False)
)

scam_rate_percents = (scam_rate_per_chain * 100)

print(scam_rate_percents.to_string())

Blockchain
BASE       100.000000
CRONO      100.000000
FANTOM     100.000000
FTM        100.000000
BSC         75.077882
POLYGON     62.500000
ARBI        58.333333
ETH         51.658768
SNOW         0.000000


##### As far as FANTOM, CRONO, BASE and SNOW blockchains have very few samples (less than 10) and are biased (only scam tokens for BASE, CRONO and FANTOM and only normal samples for SNOW), these blockchains will be removed from the dataset to avoid overfitting.

In [6]:
# Drop blockchains with number of samples less than 1%

min_samples = 1.0
chains_to_keep = blockchain_distribution[blockchain_distribution >= min_samples].index

print("Blockchains to keep:")
print(list(chains_to_keep))

print("\nDropped blockchains:")
print(blockchain_distribution[blockchain_distribution < min_samples].to_string())

data = data[data['Blockchain'].isin(chains_to_keep)].copy()


Blockchains to keep:
['ETH', 'BSC', 'POLYGON', 'ARBI']

Dropped blockchains:
Blockchain
FANTOM    0.4
CRONO     0.2
BASE      0.2
FTM       0.1
SNOW      0.1


In [7]:
# Dataset after dropping blockchains with small number of samples

print(data.shape)
data.head(5)

(990, 27)


,Project Title,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Sign,first deposits,Blockchain Type,Smart Contract (online),smart Contract (offline),website,x profile,class,project starting date,project end date,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
0,HyperVerse Token (HVT),7.650000e+00,1.500000e-01,9.100000e-06,8.000000e-07,BSC,623909,25752,9.537265e-03,4.929203e-02,166600000000100,HVT,44,POSA,sourse code,CODE,https://thehyperverse.net/index.html,https://twitter.com/HyperVerse6,scam,2022-01-27,2023-07-14,148,148,29,8,74,47
1,Fintoch,1.795000e-11,1.907000e-11,1.727000e-10,1.769000e-10,BSC,"1,492,842","147,791",2.487228e+03,4.503032e+07,34403.74594,BEP-20 TOKEN*,1,POSA,sourse code,CODE,https://web.archive.org/web/20230603123631/htt...,NaN,scam,2022-07-12,2023-06-19,820,167,4530,1590,48,4
2,Flare Token,1.955000e-03,5.519000e-04,4.158000e-04,2.838000e-04,BSC,"184,694",15173,8.901714e+14,6.695544e+17,10000000000,Flare,53,POSA,sourse code,CODE,https://pipeflare.io/,https://x.com/MetaFlareToken,scam,2021-10-24,2022-11-24,421,156,6,5,1710,1150
3,Safuu Protocol,2.070000e+02,2.110000e+02,7.000000e+01,2.400000e+01,BSC,"275,530",151979,2.457331e+10,8.139816e+14,61634066.59803,SAFUU,2,POSA,sourse code,CODE,https://safuu.com/,https://x.com/safuuxofficial,scam,2022-02-03,2022-08-13,327,323,0,0,5,4
4,SCT,2.986000e-01,1.659000e-01,1.831000e-01,1.552000e-01,BSC,8445,"6,126\n",4.252172e+06,1.410127e+05,"42,896,736.739367",SCT,52,POSA,sourse code,CODE,https://supercells.jp/en/,https://x.com/scttoken,scam,2023-02-27,2024-07-09,4990000,345000,6,3,1830,594


In [8]:
# Display blockchain distribution after dropping blockchains with small number of samples

blockchain_distribution = data['Blockchain'].value_counts(normalize=True) * 100
print(blockchain_distribution.to_string())

Blockchain
ETH        63.939394
BSC        32.424242
POLYGON     2.424242
ARBI        1.212121


##### Create a test set and a validation test

In [10]:
# Create a test set and a validation set from the raw data to avoid data leakage
# A split is temporal, preserving project period balance (based on start date), so that both old and new projects are presented in each split

# Validation set size is about 10,5% and test set size is about 24,5% from the whole data set

from sklearn.model_selection import train_test_split

# Convert project starting date to datetime and extract year
data['project starting date'] = pd.to_datetime(data['project starting date'], errors='coerce')
data['project starting year'] = data['project starting date'].dt.year

# Group years into broader periods
def assign_project_period(year):
    if year < 2021:
        return 'before_2021'
    elif year in [2021, 2022, 2023]:
        return str(year)
    elif year in [2024, 2025]:
        return '2024_2025'
    else:
        return 'unknown'

data['project period'] = data['project starting year'].apply(assign_project_period)

# Define seed
seed = 7

# Combined stratification label for the full dataset (class and project period)
combined_labels = data['class'].astype(str) + '|' + data['project period'].astype(str)

# Split the data first on train set and set for test and validation
train_set, test_and_val_set = train_test_split(data, test_size=0.35, random_state=seed, stratify=combined_labels)

# Combined stratification label for test_and_val_set (class and project period)
subset_combined_labels = (test_and_val_set['class'].astype(str) + '|' + test_and_val_set['project period'].astype(str))

#Split the part for test and validation into test set and validation set
test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=subset_combined_labels)

# Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

# Create a list of sets to perform further feature engineering on all subsets of data
data_sets = [train_set, test_set, val_set]

train_set.head(5)

Training set shape: (643, 29)
Test set shape: (242, 29)
Validation set shape: (105, 29)


,Project Title,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Sign,first deposits,Blockchain Type,Smart Contract (online),smart Contract (offline),website,x profile,class,project starting date,project end date,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2),project starting year,project period
124,GameFi (GAFI),439.652,66.671,12.942,9.015,BSC,1272,595,4.020320e+14,4.891000e+08,500000000,GAFI,49.793,POSA,sourse code,CODE,https://gamefi.org/,https://twitter.com/GameFi_Official,scam,2021-09-19,2023-06-15,905,479,5580,3950,293,38,2021,2021
475,MetaSwap (MSC),326.330,52.760,7.070,4.460,BSC,289969,14019,5.862138e+05,3.994789e+09,100000,MSC,32.35,POSA,sourse code,CODE,https://metaswap.cx/,https://twitter.com/METASWAPBSC,scam,2022-02-26,2024-10-15,6,4,5,5,15,467,2022,2022
600,Render Token (RNDR),1.939,6.750,2.730,13.630,ETH,1289427,83234,1.500177e+12,2.919331e+16,532450920.6554,RNDR,0.08256,POS,sourse code,CODE,https://rendernetwork.com/,https://x.com/rendernetwork,normal,2020-06-24,2024-11-09,24400,6120,1,0,16200,2670,2020,before_2021
867,Aurox Token (URUS),285.000,85.000,30.800,12.000,ETH,45961,20295,4.774667e+06,4.139753e+09,1000000,URUS,14,POS,sourse code,CODE,https://getaurox.com/,https://twitter.com/getaurox,normal,2021-03-21,2024-12-01,264,240,9,9,64,8,2021,2021
943,Neurashi (NEI),0.022,0.018,0.007,0.012,BSC,254117,126970,2.307539e+15,7.228413e+18,45000000000,NEI,0.022,POSA,sourse code,CODE,https://neurashi.com/,https://twitter.com/Neurashi,normal,2024-01-03,2025-01-17,7,4,4,2,72,42,2024,2024_2025


In [11]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

Overlap between train and test: 0
Overlap between train and validation: 0
Overlap between test and validation: 0


In [ ]:
# Delete columns that are not useful for further analysis

data.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date', 'project starting year', 'project period'], inplace=True)

# Check the transformation
data.head(10)

## Raw data analysis

#### Performed first on raw data to get overall idea of what the dataset contains
##### All further exploratory operations except checking general info should be done only on the train set to avoid data leakage.
##### Analysis of skew and correlation is performed only on numeric features first. Categorial features are initially analysed separately, then encoded during pre-processing.

In [ ]:
# Encode class label

from sklearn.preprocessing import LabelEncoder

for set in data_sets:
    le = LabelEncoder()
    set['class'] = set['class'].map({'normal': 0, 'scam': 1})

train_set

In [ ]:
# Check the data description

description = train_set.describe()
description

In [12]:
# Check the balance between classes in the training set

class_counts_train = data['class'].value_counts()
class_percent_train = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts_train.to_string(), "\nIn %:", class_percent_train.to_string())

Counts: class
scam      590
normal    400 
In %: class
scam      59.59596
normal    40.40404


In [ ]:
print(train_set.dtypes)

In [ ]:
# Transfer all data to numeric values

#TODO: reference from notes

# Columns with object datatype
cols_to_clean = [
    'the number of Transactions',
    'Token concentration ratio per holder',
    'Token balance',
    'first deposits',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)'
]

# Apply to all data splits
data_sets = [train_set, test_set, val_set]

for i, d_set in enumerate(data_sets):
    for col in cols_to_clean:
        data_sets[i][col] = pd.to_numeric(
            d_set[col].astype(str)
                   .str.replace('\xa0', '', regex=False)        # Remove non-breaking whitespaces
                   .str.replace(',', '', regex=False)           # Strip comas separating numeric values
                   .str.strip(),                                # Remove surrounding whitespaces
            errors='coerce'                                     # If cannot parse, put NaN
        )

train_set, test_set, val_set = data_sets

# Verify
print(train_set[cols_to_clean].dtypes)
print(train_set[cols_to_clean].isnull().sum())
print(train_set[cols_to_clean].describe())

In [ ]:
# Check for skew for numeric columns

numeric_cols = train_set.select_dtypes(include=['number']).columns
skew = train_set[numeric_cols].skew()

skew

### Visualisation

##### Based on CSM010-2024-APR, Topic 2, Lab.2.15. Analysing data

In [ ]:
# Histograms

train_set.hist(figsize=[30, 30])
pyplot.show()

In [ ]:
# Density plots

train_set.plot(kind='density', subplots=True, layout=(8,7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Box and Whisker Plots

train_set.plot(kind='box', subplots=True, layout=(8,7), sharex=False, sharey=False,figsize=[30, 30])
pyplot.show()

##### Data is extremely skewed and there are extreme outliers, thus, data requires pre-processing.

In [ ]:
# Search for correlations of numeric features

correlations = train_set[numeric_cols].corr(method='pearson')

correlations

In [ ]:
# Correlation Matrix Plot

fig = pyplot.figure(figsize=[30, 30])
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

ticks = np.arange(len(correlations.columns))

ax.set_xticks(ticks)
ax.set_yticks(ticks)

short_names = list(train_set[numeric_cols].columns)

ax.set_xticklabels(short_names)
ax.set_yticklabels(short_names)

pyplot.xticks(rotation=90)     # https://www.geeksforgeeks.org/how-to-rotate-x-axis-tick-label-text-in-matplotlib/?ysclid=lxdkiwmkrh456759424

pyplot.show()

In [ ]:
# Scatter plot Matrix
# TODO: make it more readable

axes = pd.plotting.scatter_matrix(train_set[numeric_cols], figsize=(20, 20))

for ax_row in axes:
    for ax in ax_row:
        ax.xaxis.label.set_rotation(90)
        ax.yaxis.label.set_rotation(0)
        ax.xaxis.label.set_fontsize(8)
        ax.yaxis.label.set_fontsize(8)
        ax.tick_params(axis='both', labelsize=5)

        # Shift y labels left
        ax.yaxis.set_label_position("left")
        ax.yaxis.label.set_horizontalalignment('right')
        ax.yaxis.label.set_x(-0.2)

pyplot.show()

##### Correlation matrix shows strong correlation (almost 1.0) between 'first deposits' and 'Q1'. Manual analysis reveal that 'first deposits' column is likely to be corrupted, since a lot of samples have the same float value here as in 'Q1', which does not make sense (there should be an integer).
##### There is also a strong correlation between price features (Q1, Q2, Q3 and Q4), thus, it is considered to drop three of them.

##### Based on results of data analysis on raw data, some pre-processing is required. In three following sections different levels of pre-processing will be tried. At first, minimal pre-processing will be applied (handling missing values, encode categorical features). Secondly, heavy skew and extreme outliers will be handled. Then model specific pre-processing will be made for each of intended models.

##### Then the following scenarios will be evaluated:
##### - train all models on raw version of data with minimal pre-processing
##### - train all models on moderately pre-processed version of data (which deals with skew and extreme outliers)
##### - train all models on model-specific version of data.

## Minimal pre-processing of data

#### Based on results of data analysis on raw data, some pre-processing is required in order to run most models. In this section, minimal pre-processing will be applied (handling missing values, encoding categorical features) in order to make the dataset suitable for model training.

In [ ]:
# Remove 'first deposits' manually, since it seems to be corrupted (in majority cases it reproduces Q1 figures, which are prices, not deposits count)
# TODO: consider to replace 'first deposits' (number of initial liquidity deposit events)??

for data_set in data_sets:
    data_set.drop(columns=['first deposits'], inplace=True)

train_set

In [ ]:
# Handle missing values (for numeric 'median' is used due to heavy skew, for categorical values 'most frequent' is used)
# SimpleImputer is trained for all numeric and categorical columns (not only for ones with missing values) to be able to handle possible future missing values.

# Based on: Géron, A. "End-to-end Machine Learning Project" in 'Hands-on' machine learning with Scikit-Learn, Keras & Tensorflow. (O'Reilly Media, Inc, 2019) 2nd edition.
# TODO: reference to docs

from sklearn.impute import SimpleImputer

# Define numeric and categorical columns
numeric_cols = train_set.select_dtypes(include=['number']).columns.tolist()
categorical_cols = train_set.select_dtypes(exclude=['number']).columns.tolist()

# Create inputers and fit to train set
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
num_imputer.fit(train_set[numeric_cols])
cat_imputer.fit(train_set[categorical_cols])

# Apply to train, test, val sets
for data_set in [train_set, test_set, val_set]:
    data_set[numeric_cols] = num_imputer.transform(data_set[numeric_cols])
    data_set[categorical_cols] = cat_imputer.transform(data_set[categorical_cols])

# Verify it works (no missing values left)
print("\nRemaining missing values per column in train_set:")
print(train_set.isnull().sum())

In [ ]:
# Determine unique categories for 'Blockchain' and 'Blockchain Type' columns

print(data_sets[0]['Blockchain'].unique())
print(data_sets[0]['Blockchain Type'].unique())

In [ ]:
# Encode categorical features ('Blockchain', 'Blockchain Type') to make all features numeric.
# OneHotEncoder was chosen, since the number of categories is small (4 for 'Blockchain' and 3 for 'Blockchain Type',
# and OneHotEncoder prevents algorithms to treat close values more similar than distant ones.

# Based on: Based on: Géron, A. "End-to-end Machine Learning Project" in 'Hands-on' machine learning with Scikit-Learn, Keras & Tensorflow. (O'Reilly Media, Inc, 2019) 2nd edition.
# TODO: Reference to docs OneHotEncoder

from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['Blockchain', 'Blockchain Type']
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoder.fit(train_set[categorical_cols])

# Apply to each split
for data_set in [train_set, val_set, test_set]:
    cat_array = cat_encoder.transform(data_set[categorical_cols])
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols)
    cat_df = pd.DataFrame(cat_array, columns=cat_feature_names, index=data_set.index)

    # Drop original categorical columns
    data_set.drop(columns=categorical_cols, inplace=True)

    # Add encoded columns
    for col in cat_df.columns:
        data_set[col] = cat_df[col]

# Verify it works (all columns are numeric)
print(train_set.dtypes)

In [ ]:
# Determine how many extreme outliers are in dataset
# Several numeric features contains extreme values like 10ˆ40 or 10ˆ60, that cannot be handled by estimators and produce an error (Input X contains infinity or a value too large for dtype('float32').

# Reference: https://numpy.org/devdocs/reference/generated/numpy.finfo.html

MAX_F32 = np.finfo(np.float32).max

def count_above_f32(df, name):
    arr = df[numeric_cols].to_numpy()
    above = np.abs(arr) > MAX_F32
    n_above = np.count_nonzero(above)
    total = arr.size

    print(
        f"{name}: values above float32 max: {n_above} "
        f"({n_above / total:.6%})"
    )

count_above_f32(train_set, "train_minimal")
count_above_f32(val_set,   "val_minimal")
count_above_f32(test_set,  "test_minimal")

In [ ]:
# Clip extreme values to make data suitable for model training (applied to copies of data, since this step is not relevant
# after transformation (see next section)).
# Clipping is performed for values above float32 datatype capacity.

# Reference: M. Kuhn and K. Johnson, Applied Predictive Modeling. New York, NY, USA: Springer, 2013.

train_minimal = train_set.copy()
val_minimal   = val_set.copy()
test_minimal  = test_set.copy()

numeric_cols_minimal = train_minimal.select_dtypes(include=['number']).columns.tolist()

# Clip everything to float32 capacity
for data_set in [train_minimal, val_minimal, test_minimal]:
    data_set[numeric_cols_minimal] = data_set[numeric_cols_minimal].clip(upper=MAX_F32)

# Verify
for name, data_set in zip(
    ['train_minimal', 'val_minimal', 'test_minimal'],
    [train_minimal, val_minimal, test_minimal]
):
    arr = data_set[numeric_cols_minimal].to_numpy()
    print(f"{name}: any > MAX_F32? {(np.abs(arr) > MAX_F32).any()}")

In [ ]:
# Separate features from the target variable for minimally pre-processed dataset
# Extract X and Y from train, test and validation sets after minimal pre-processing

# TODO: reference to AML tutorial

X_train_minimal = train_minimal.drop(columns=['class']).values
Y_train_minimal = train_minimal['class'].values

X_test_minimal = test_minimal.drop(columns=['class']).values
Y_test_minimal = test_minimal['class'].values

X_val_minimal = val_minimal.drop(columns=['class']).values
Y_val_minimal = val_minimal['class'].values

## General moderate pre-processing

#### Due to extreme skew, some data transformation will be tried in order to improve performance of models.

#### Data after this step of pre-processing also will be used for plotting to assess the effect of pre-processing on data.

In [ ]:
# Transform data to deal with skew and outliners
# For columns with heavy skew log1p transform and clipping is applied (clip values above the 99.5th percentile)

# Reference: M. Kuhn and K. Johnson, Applied Predictive Modeling. New York, NY, USA: Springer, 2013.
# Reference: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PowerTransformer.html

# TODO: to report / notes: log1p transformation has been attempted, resulting in reducing skew to small to moderate values, but several features were still
# TODO: heavily skewed. Thus, PowerTransformer has been applied to find a power that is the most effective in reducing skew for the dataset.

from sklearn.preprocessing import PowerTransformer

skewed_cols = [
    'MaxPrice (Quarter 1)',
    'MaxPrice (Quarter 2)',
    'MaxPrice (Quarter 3)',
    'MaxPrice (Quarter 4)',
    'Total Variance',
    'Variance of holders with more than 1% tokens',
    'Token balance',
    'Token concentration ratio per holder',
    'the number of Transactions',
    'Google results for project title (first day)',
    'Google results for project title (project duration/2)',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)',
    'Google results for project x profile (first days)',
    'Google results for project x profile (duration/2)'
]

# Yeo-Johnson has been chosen, since it handles zeros safely. At this stage, no scaling is applied
pt = PowerTransformer(method='yeo-johnson', standardize=False)
pt.fit(train_set[skewed_cols].astype(float).values)

for data_set in [train_set, val_set, test_set]:
    data_set[skewed_cols] = pt.transform(data_set[skewed_cols].astype(float).values)

# Assess skew after transformation
skew = train_set.skew()
skew

In [ ]:
# Make snapshots of moderately pre-processed data for further experimentation

train_moderate = train_set.copy()
val_moderate   = val_set.copy()
test_moderate  = test_set.copy()

In [ ]:
# Separate features from the target variable for moderately pre-processed dataset
# Extract X and Y from train, test and validation sets after moderate pre-processing

# TODO: reference to AML tutorial

X_train_moderate = train_moderate.drop(columns=['class']).values
Y_train_moderate = train_moderate['class'].values

X_test_moderate = test_moderate.drop(columns=['class']).values
Y_test_moderate = test_moderate['class'].values

X_val_moderate = val_moderate.drop(columns=['class']).values
Y_val_moderate = val_moderate['class'].values

### Visualisation after data transformation

In [ ]:
# Histograms after handling skew

train_moderate.hist(figsize=[30, 30])
pyplot.show()

In [ ]:
# Density plots after handling skew

train_moderate.plot(kind='density', subplots=True, layout=(8,7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Box and Whisker Plots after handling skew

train_moderate.plot(kind='box', subplots=True, layout=(8,7), sharex=False, sharey=False,figsize=[30, 30])
pyplot.show()

In [ ]:
# Search for correlations of numeric features after handling skew

correlations = train_moderate[numeric_cols].corr(method='pearson')

correlations

In [ ]:
# Correlation Matrix Plot after handling skew

fig = pyplot.figure(figsize=[30, 30])
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

numeric_cols = correlations.columns

ticks = np.arange(len(numeric_cols))

ax.set_xticks(ticks)
ax.set_yticks(ticks)

short_names = list(numeric_cols)

ax.set_xticklabels(short_names)
ax.set_yticklabels(short_names)

pyplot.xticks(rotation=90)     # https://www.geeksforgeeks.org/how-to-rotate-x-axis-tick-label-text-in-matplotlib/?ysclid=lxdkiwmkrh456759424

pyplot.show()

In [ ]:
# Scatter plot Matrix after handling skew

# Reference: https://matplotlib.org/3.10.3/gallery/text_labels_and_annotations/align_ylabels.html

axes = pd.plotting.scatter_matrix(train_moderate[numeric_cols], figsize=(20, 20))

for ax_row in axes:
    for ax in ax_row:
        ax.xaxis.label.set_rotation(90)
        ax.yaxis.label.set_rotation(0)
        ax.xaxis.label.set_fontsize(8)
        ax.yaxis.label.set_fontsize(8)
        ax.tick_params(axis='both', labelsize=5)

        # Shift y labels left
        ax.yaxis.set_label_position("left")
        ax.yaxis.label.set_horizontalalignment('right')
        ax.yaxis.label.set_x(-0.2)

pyplot.show()

## Model specific pre-processing

##### Since models from different families will be trained on data, data requires some additional pre-processing depending on model trained. Some algorithms perform better on rescaled data (e.g. K-Nearest Neighbors or Linear regression), whereas others (like Random Forest) may degrade after rescaling. In this section, model specific pre-processing will be handled and then all versions of data (after minimal pre-processing, after moderate pre-processing and after specific pre-processing) will be used to train models and evaluate performance.

## Training models

### Random Forest

##### Random Forest usually handles outliers and values with different scales well, and applying normalising or re-scaling techniques do not improve performance. Thus, Random Forest model will be trained on two versions of data: after minimal pre-processing and after transformation.

In [ ]:
# Train and evaluate Random Forest algorithm on minimally pre-processed dataset (with skew remaining)

# TODO: get rid of repetitive code, introduce some functions and nicely formatted comparative output

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=seed
)

rf.fit(X_train_minimal, Y_train_minimal)

Y_test_pred  = rf.predict(X_test_minimal)
Y_test_proba = rf.predict_proba(X_test_minimal)[:, 1]

print("Test classification report:\n")
print(classification_report(Y_test_minimal, Y_test_pred, digits=3))

test_auc = roc_auc_score(Y_test_minimal, Y_test_proba)
print(f"Test ROC AUC: {test_auc:.3f}")

In [ ]:
# Train and evaluate Random Forest algorithm on moderately pre-processed dataset

rf.fit(X_train_moderate, Y_train_moderate)

Y_test_pred  = rf.predict(X_test_moderate)
Y_test_proba = rf.predict_proba(X_test_moderate)[:, 1]

print("Test classification report:\n")
print(classification_report(Y_test_moderate, Y_test_pred, digits=3))

test_auc = roc_auc_score(Y_test_moderate, Y_test_proba)
print(f"Test ROC AUC: {test_auc:.3f}")

## Feature importance assessment

In [ ]:
# TODO: assess feature importance after training models to understand what features were most predictive